<a href="https://colab.research.google.com/github/SalmaEzz10/NLPCloud/blob/main/CloudNLPUpdatedFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets seqeval arabert accelerate -q evaluate emoji regex

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 19.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 13.9 MB/s eta 0:00:00


In [ ]:
import json, random
import emoji
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset, DatasetDict
import evaluate

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
df = pd.read_csv('datasetCloud_fixed.csv')

In [ ]:
df.head(5)

,address,annotations,bio
0,دمياط الرحاب,"{""CITY"": ""دمياط"", ""AREA"": ""الرحاب""}","[[""دمياط"", ""B-CITY""], [""الرحاب"", ""B-AREA""]]"
1,المعصره شقه ٥ قريب من محطه المترو عماره ١٢ شار...,"{""AREA"": ""المعصره"", ""STREET"": ""الهرم"", ""LANDMA...","[[""المعصره"", ""B-AREA""], [""شقه"", ""O""], [""٥"", ""B..."
2,محافظه الغردقه عقار نفرتيتي شارع عباس العقاد,"{""CITY"": ""الغردقه"", ""STREET"": ""عباس العقاد"", ""...","[[""محافظه"", ""O""], [""الغردقه"", ""B-CITY""], [""عقا..."
3,عقار الرياحين ميت غمر طابق ١٠,"{""CITY"": ""ميت غمر"", ""BUILDING"": ""الرياحين"", ""F...","[[""عقار"", ""O""], [""الرياحين"", ""B-BUILDING""], [""..."
4,سموحه في الدقهليه,"{""CITY"": ""الدقهليه"", ""AREA"": ""سموحه""}","[[""سموحه"", ""B-AREA""], [""في"", ""O""], [""الدقهليه""..."


In [ ]:
print(df.isnull().sum())
print("\n duplicated")
print(df.duplicated().sum())

address        0
annotations    0
bio            0
dtype: int64

 duplicated
2


In [ ]:
# df = df.dropna()
df = df.drop_duplicates()
df = df.reset_index(drop=True)

print(df.duplicated().sum())

0


In [ ]:
def removeEmojis(text):
    return emoji.replace_emoji(str(text), replace='')

In [ ]:
test = "🙂"
removeEmojis(test)

''

In [ ]:
def removePunct(text):
    punct = "`؛;_ـ،:.-'|,"
    return re.sub(f"[{re.escape(punct)}]", "", text)

In [ ]:
test = "|-"
removePunct(test)

''

In [ ]:
def changeEnglishToArabic(text):
    mapping = {
        "floor": "دور",
        "st": "شارع",
        "street": "شارع",
        "apartment": "شقه",
        "apt": "شقه",
    }

    for eng, ar in mapping.items():
        text = re.sub(rf"\b{eng}\b", ar, text, flags=re.IGNORECASE)

    return text

In [ ]:
test = "FLOOR"
changeEnglishToArabic(test)

'دور'

In [ ]:
def changeNumEngToArabic(text):
    eng = "0123456789"
    arb = "٠١٢٣٤٥٦٧٨٩"
    return text.translate(str.maketrans(eng, arb))

In [ ]:
test = "12"
changeNumEngToArabic(test)

'١٢'

In [ ]:
def convertCharToWord(text):
    mapping = {
        "ش": "شارع"
    }

    for x, y in mapping.items():
        text = re.sub(rf"\b{x}\b", y, text)

    return text

In [ ]:
test = "ش"
convertCharToWord(test)

'شارع'

In [ ]:
def removeDiacritics(text):
    return re.sub(r'[\u064B-\u0652]', '', text)

In [ ]:
# def normalizeSomeChar(text):
#     src = "اأإآيىگكؤوةه"
#     dst = "ااااييككووهه"
#     return text.translate(str.maketrans(src, dst))
def normalizeSomeChar(text):
    text = re.sub("[أإآ]", "ا", text)
    text = re.sub("ى", "ي", text)
    return text

In [ ]:
test = "ش أحمد كمأل"
normalizeSomeChar(test)

'ش احمد كمال'

In [ ]:
# def dropDuplicateSafeChars(text):
#     return re.sub(r"([^\d])\1+", r"\1", text)

In [ ]:
def numArabicApt(text):
    mapping = {
        "واحد":"١",
        "الاولي":"١",
        "اثنين":"٢",
        "اتنين":"٢",
        "الثانيه":"٢",
        "التانيه":"٢",
        "ثلاثه":"٣",
        "تلاته":"٣",
        "الثالثه":"٣",
        "اربعه":"٤",
        "الرابعه":"٤",
        "خمسه":"٥",
        "سته":"٦",
        "سبعه":"٧",
        "ثمانيه":"٨",
        "تمانيه":"٨",
        "تسعه":"٩",
        "عشره":"١٠"
    }

    for w, n in mapping.items():
        text = re.sub(rf"\b{w}\b", n, text)

    return text

In [ ]:
test = "شقه خمسه"
numArabicApt(test)

'شقه ٥'

In [ ]:
def numArabicFloor(text):
    mapping = {
        "الاول":"١",
        "الثاني":"٢",
        "التاني":"٢",
        "الثالث":"٣",
        "التالت":"٣",
        "الرابع":"٤",
        "الخامس":"٥",
        "السادس":"٦",
        "السابع":"٧",
        "الثامن":"٨",
        "التامن":"٨",
        "التاسع":"٩",
        "العاشر":"١٠"
    }

    for w, n in mapping.items():
        text = re.sub(rf"\b{w}\b", n, text)

    return text

In [ ]:
def preprocessing(text):
    text = str(text)
    text = removeEmojis(text)
    text = changeEnglishToArabic(text)
    text = removePunct(text)
    text = convertCharToWord(text)
    text = changeNumEngToArabic(text)
    text = removeDiacritics(text)
    text = normalizeSomeChar(text)
    # text = dropDuplicateSafeChars(text)
    text = numArabicApt(text)
    text = numArabicFloor(text)
    text = " ".join(text.split())

    return text

In [ ]:
test = "ش محمد سيد لحمد  تقاطع  ش فيضي 😂   حلوان القاهره"
preprocessing(test)

'شارع محمد سيد لحمد تقاطع شارع فيضي حلوان القاهره'

In [ ]:
print(df.loc[30,'address'])
print(df.loc[30,'annotations'])
print(df.loc[30,'bio'])

البحيره شقه ١٨ شارع طلعت حرب ميت عقبه بجوار الكليه عماره ١٠
{"CITY": "البحيره", "AREA": "ميت عقبه", "STREET": "طلعت حرب", "LANDMARK": "الكليه", "BUILDING": "١٠", "APARTMENT": "١٨"}
[["البحيره", "B-CITY"], ["شقه", "O"], ["١٨", "B-APARTMENT"], ["شارع", "O"], ["طلعت", "B-STREET"], ["حرب", "I-STREET"], ["ميت", "B-AREA"], ["عقبه", "I-AREA"], ["بجوار", "O"], ["الكليه", "B-LANDMARK"], ["عماره", "O"], ["١٠", "B-BUILDING"]]


In [ ]:
entities = ['CITY', 'AREA', 'MINI_AREA', 'NUM_STREET', 'MAIN_STREET',
            'STREET', 'LANDMARK', 'BUILDING', 'FLOOR', 'APARTMENT']

labelList = ['O']
for ent in entities:
    labelList.append(f'B-{ent}')
    labelList.append(f'I-{ent}')

LABEL2ID = {lbl: i for i, lbl in enumerate(labelList)}
ID2LABEL = {i: lbl for lbl, i in LABEL2ID.items()}

print(f'{len(labelList)} labels')
print(f'{labelList}')

21 labels
['O', 'B-CITY', 'I-CITY', 'B-AREA', 'I-AREA', 'B-MINI_AREA', 'I-MINI_AREA', 'B-NUM_STREET', 'I-NUM_STREET', 'B-MAIN_STREET', 'I-MAIN_STREET', 'B-STREET', 'I-STREET', 'B-LANDMARK', 'I-LANDMARK', 'B-BUILDING', 'I-BUILDING', 'B-FLOOR', 'I-FLOOR', 'B-APARTMENT', 'I-APARTMENT']


In [ ]:
def parseBIORow(BIOAddress):
    try:
        bio = json.loads(BIOAddress)
        tokens = [pair[0] for pair in bio]
        labels = [pair[1] for pair in bio]
        return tokens, labels
    except:
        return None, None

In [ ]:
records = []
skipped = 0

for _, row in df.iterrows():
    tokens, labels = parseBIORow(row['bio'])

    if tokens is None or len(tokens) == 0:
        skipped += 1
        continue

    if len(tokens) != len(labels):
        skipped += 1
        continue

    if any(lbl not in LABEL2ID for lbl in labels):
        skipped += 1
        continue

    records.append({'tokens': tokens, 'labels': labels})

print(f'Valid records: {len(records)}')
print(f'Skipped: {skipped}')

Valid records: 9999
Skipped: 0


In [ ]:
random.shuffle(records)
n = len(records)
n_train = int(0.80 * n)
n_val = int(0.10 * n)

train_records = records[:n_train]
val_records = records[n_train:n_train + n_val]
test_records = records[n_train + n_val:]

In [ ]:
modelName = 'aubmindlab/bert-base-arabertv02'
tokenizer = AutoTokenizer.from_pretrained(modelName)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
def tokenizeAndAlignLabels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=128,
        is_split_into_words=True,
        padding=False,
    )
    aligned_labels = []
    for i, label_seq in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)

            elif word_id != prev_word_id:
                label_ids.append(LABEL2ID[label_seq[word_id]])

            else:
                label_ids.append(-100)

            prev_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized['labels'] = aligned_labels

    return tokenized

In [ ]:
def recordsToHF(records_list):
    return HFDataset.from_dict({
        'tokens': [r['tokens'] for r in records_list],
        'labels': [r['labels'] for r in records_list],
    })


hf_train = recordsToHF(train_records).map(tokenizeAndAlignLabels, batched=True, remove_columns=['tokens', 'labels'])
hf_val = recordsToHF(val_records).map(tokenizeAndAlignLabels, batched=True, remove_columns=['tokens', 'labels'])
hf_test = recordsToHF(test_records).map(tokenizeAndAlignLabels, batched=True, remove_columns=['tokens', 'labels'])

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    modelName,
    num_labels=21,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model = model.to('cuda')

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

In [ ]:
seqeval = evaluate.load('seqeval')

def computeMetrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        true_label_row, true_pred_row = [], []
        for pred, label in zip(pred_seq, label_seq):
            if label == -100:   # ignore special tokens
                continue
            true_label_row.append(ID2LABEL[label])
            true_pred_row.append(ID2LABEL[pred])
        true_labels.append(true_label_row)
        true_preds.append(true_pred_row)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        'precision':results['overall_precision'],
        'recall':results['overall_recall'],
        'f1':results['overall_f1'],
        'accuracy':results['overall_accuracy'],
    }

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
)

In [ ]:
OUTPUT_DIR = './arabert_ner_output'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,


    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,


    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,


    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    seed=42,

    save_total_limit=2,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    data_collator=data_collator,
    compute_metrics=computeMetrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.061221,0.023346,0.992248,0.995463,0.993853,0.995012
2,0.038468,0.019176,0.994822,0.996111,0.995466,0.996157
3,0.015309,0.016921,0.994985,0.996436,0.995710,0.996647
4,0.007891,0.014873,0.995148,0.996922,0.996034,0.997056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
modelNLPCloud = './arabert_ner_best_model'
trainer.save_model(modelNLPCloud)
tokenizer.save_pretrained(modelNLPCloud)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./arabert_ner_best_model/tokenizer_config.json',
 './arabert_ner_best_model/tokenizer.json')

In [ ]:
test_results = trainer.predict(hf_test)
test_metrics = test_results.metrics

print(f'Precision:{test_metrics["test_precision"]:.4f}')
print(f'Recall:{test_metrics["test_recall"]:.4f}')
print(f'F1 Score:{test_metrics["test_f1"]:.4f}')
print(f'Accuracy:{test_metrics["test_accuracy"]:.4f}')

Precision:0.9966
Recall:0.9970
F1 Score:0.9968
Accuracy:0.9969


In [ ]:
from transformers import pipeline

ner_pipeline = pipeline(
    'ner',
    model=modelNLPCloud,
    tokenizer=modelNLPCloud,
    aggregation_strategy='first',
    device=0 if torch.cuda.is_available() else -1,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def predict_address(address: str, verbose: bool = True) -> dict:

    results = ner_pipeline(address)

    extracted = {}
    for item in results:
        entity_type = item['entity_group'].replace('B-', '').replace('I-', '')
        if entity_type == 'O':
            continue

        word = item['word'].strip()
        score = item['score']

        if entity_type not in extracted:
            extracted[entity_type] = {'value': word, 'score': score}
        else:
            extracted[entity_type]['value'] += ' ' + word
            extracted[entity_type]['score'] = min(extracted[entity_type]['score'], score)

    final_output = {}
    for e in entities:
        if e in extracted:
            final_output[e] = extracted[e]['value']
        else:
            final_output[e] = None

    if verbose:
        print(f'Address: {address}')

        for entity in entities:
            value = final_output[entity]

            if value:
                score = extracted[entity]['score']
                print(f'{entity:<12} │ {value:<30} │ conf: {score:.2%}')
            else:
                print(f'{entity:<12} │ {"--":<30} │ conf: 0.00%')

    return final_output

In [ ]:
custom_address =   "11 ش بشار بن برد, متفرع من ش مكرم عبيد, المنطقة السادسة, مدينة نصر, القاهرة."
custom_address = preprocessing(custom_address)
result = predict_address(custom_address)
print()
print(json.dumps(result, ensure_ascii=False, indent=2))

Address: ١١ شارع بشار بن برد متفرع من شارع مكرم عبيد المنطقة السادسة مدينة نصر القاهرة
CITY         │ القاهرة                        │ conf: 99.99%
AREA         │ مدينة نصر                      │ conf: 99.96%
MINI_AREA    │ المنطقة السادسة                │ conf: 98.59%
NUM_STREET   │ ١١                             │ conf: 99.98%
MAIN_STREET  │ مكرم عبيد                      │ conf: 99.94%
STREET       │ بشار بن برد                    │ conf: 99.87%
LANDMARK     │ --                             │ conf: 0.00%
BUILDING     │ --                             │ conf: 0.00%
FLOOR        │ --                             │ conf: 0.00%
APARTMENT    │ --                             │ conf: 0.00%

{
  "CITY": "القاهرة",
  "AREA": "مدينة نصر",
  "MINI_AREA": "المنطقة السادسة",
  "NUM_STREET": "١١",
  "MAIN_STREET": "مكرم عبيد",
  "STREET": "بشار بن برد",
  "LANDMARK": null,
  "BUILDING": null,
  "FLOOR": null,
  "APARTMENT": null
}
